# Quickstart: Dynamics365-like ERP/CRM KPI Pipeline

This notebook runs the **full pipeline** end-to-end:

1. Generate synthetic ERP/CRM-style tables (Accounts, Opportunities, Cases, Work Orders)
2. ETL + data quality checks
3. KPI computation
4. Baseline ML model: **SLA breach risk**
5. Generate 2 plots for the README

> Tip: This is designed to be close to what you'd do with Dynamics 365 / Dataverse exports, but using synthetic data so it's reproducible.


In [ ]:
import sys, subprocess, json
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent  # repo root when notebook is in notebooks/
print('Working dir:', BASE_DIR)
print('Python:', sys.version.split()[0])

def run(*args):
    print('▶', ' '.join(args))
    subprocess.run([sys.executable, *args], cwd=str(BASE_DIR), check=True)


In [ ]:
# 1) Generate synthetic raw tables
run('-m', 'src.generate_data')

In [ ]:
# 2) ETL (parse datetimes, joins, quality report, feature table)
run('-m', 'src.etl')

In [ ]:
# 3) Compute KPIs
run('-m', 'src.kpis')

kpi_path = BASE_DIR / 'outputs' / 'kpi_table.csv'
kpis = pd.read_csv(kpi_path)
kpis

In [ ]:
# 4) Train baseline model (SLA breach risk)
run('-m', 'src.model')

metrics = json.loads((BASE_DIR / 'outputs' / 'model_metrics.json').read_text(encoding='utf-8'))
metrics

In [ ]:
# 5) Generate visuals for README
run('-m', 'src.plots')

from IPython.display import Image, display
display(Image(filename=str(BASE_DIR / 'outputs' / 'sla_breach_by_priority.png')))
display(Image(filename=str(BASE_DIR / 'outputs' / 'opportunities_by_stage.png')))
